# Week 13: clone generation for the new speakers (Tortoise)

Generates the new attack sets from the job tables committed in the repository, then
turns them straight into the project's bundles: each clone is quality-screened,
level-normalised to -23 dBFS at 16 kHz (clean), and rendered through the G.711 phone
line at 20 dB (channel).

| Set | Tool | Speakers | Clips | Role | Time on 2 x T4 |
|---|---|---|---|---|---|
| CM06 | Tortoise-TTS | 60 new test speakers | 240 | unseen tool, new speakers (test) | ~3-5 h |

Tortoise needs minutes per clip on a T4 (its key-value cache is on, which only changes
the speed), which is why this set is small and runs on its own.

Each tool runs in its own Python 3.11 environment (made with `uv`), because Kaggle's
Python is 3.13 and the tools pin libraries that conflict with each other. The jobs of
each tool are split across both T4 GPUs.

**How to run.** Inputs: `saikrishnareddy9/codemix-extension-bundle` and `saikrishnareddy9/mentor-signoff` (the signed ethics note; generation refuses to start without it). Settings: **GPU T4 x2**,
Internet **On**. Then **Save Version -> Save & Run All (Commit) -> Save**. That
runs in the background: the tab can be closed and the laptop shut down. Nothing needs
changing in the cells.

**Getting the results.** The only file kept in `/kaggle/working` is `generation_B_results.zip`, so it is the
version's whole output. When the version finishes, open the notebook's page -> **Output**
-> `generation_B_results.zip` -> Download. The zip is rewritten after every tool, and the run stops itself
at 10.5 hours and keeps what it made, so a result is always saved inside Kaggle's
12-hour limit. (Run All in the editor also works, but then the tab must stay open.)

## 1. Settings

In [ ]:
PLAN = [  # (tool, pack in the dataset, pool the job table must contain, attack id, split, language)
    ("tortoise", "tortoise_eval_ext", "eval_ext", "cm06", "eval", "hi-en"),
]
SHARDS = 2                 # one process per T4
ZIP_NAME = "generation_B_results.zip"
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH = "week13-krishna-generation-and-paper"
TORCH = "torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121"
PACKAGES = {               # each tool's environment, beside torch
    "xtts_v2": "coqui-tts==0.25.1 click numpy==1.26.4 pandas soundfile pyyaml",  # spaCy imports click
    "tortoise": "tortoise-tts==3.0.0 numpy==1.26.4 pandas soundfile pyyaml",
    # f5-tts pins numpy 1.26; newer pyarrow needs numpy 2, so pyarrow and datasets are pinned too
    "indicf5": "'f5-tts @ git+https://github.com/ai4bharat/IndicF5.git' transformers==4.49.0 numpy==1.26.4 pyarrow==19.0.1 datasets==3.6.0 pandas soundfile pyyaml",
}

## 2. Setup: clone, inputs, one environment per tool

In [ ]:
import glob, json, os, shutil, subprocess, sys, time, traceback, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")    # the saved output: only the results zip goes here
TEMP = Path("/kaggle/temp")       # code, tool environments and raw clones: not saved
try:
    TEMP.mkdir(parents=True, exist_ok=True)
    (TEMP / ".writable").write_text("ok")
except OSError:
    TEMP = Path("/tmp/w13")
    TEMP.mkdir(parents=True, exist_ok=True)
REPO = TEMP / "codemix-deepfake-detection"
OUT = TEMP / "generated"          # raw clones, per pack
BUNDLE = TEMP / "gen_bundle"      # what the zip carries: clean/, channel/, manifests/, qa/, logs/
DEADLINE = time.time() + 10.5 * 3600  # Kaggle stops a run at 12 h; leave time to bundle and save
FAILURES = []


def sh(cmd, check=True, env=None):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    if r.stdout.strip():
        print(r.stdout[-2000:])
    if r.returncode and r.stderr.strip():
        print(r.stderr[-3000:])
    if check and r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r


def failed(where, detail):
    FAILURES.append({"where": where, "detail": str(detail)[-3000:]})
    print(f"\n!!!!!! FAILED: {where}\n{str(detail)[-1500:]}\n!!!!!! recorded; the run continues\n", flush=True)


def save_progress():
    """Zip the bundles, manifests, QA and logs (not the raw clones) into ZIP_NAME."""
    staging = WORK / (ZIP_NAME + ".tmp")
    with zipfile.ZipFile(staging, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(BUNDLE.rglob("*")):
            if f.is_file():
                zf.write(f, f.relative_to(BUNDLE.parent))
    staging.replace(WORK / ZIP_NAME)


os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
for attempt in range(3):
    shutil.rmtree(REPO, ignore_errors=True)
    if subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                       f"https://{REPO_HOST}", str(REPO)]).returncode == 0:
        break
    time.sleep(20)
else:
    raise SystemExit("Could not clone the repository. Is Internet switched On?")
sys.path.insert(0, str(REPO))
os.chdir(REPO)
for d in (OUT, BUNDLE / "manifests", BUNDLE / "qa", BUNDLE / "logs"):
    d.mkdir(parents=True, exist_ok=True)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], text=True, capture_output=True).stdout.strip()

# The ethics gate: every generator refuses to load without the signed mentor note.
signoff = glob.glob("/kaggle/input/**/mentor_signoff*.pdf", recursive=True)
if not signoff:
    raise SystemExit("Attach saikrishnareddy9/mentor-signoff (Add Input): generation needs the signed ethics note")
(REPO / "docs" / "ethics").mkdir(parents=True, exist_ok=True)
shutil.copy2(signoff[0], REPO / "docs" / "ethics" / Path(signoff[0]).name)
from src.data.ethics_gate import signoff_status
print(signoff_status().describe())

PACKS = {Path(p).parent.name: Path(p).parent
         for p in glob.glob("/kaggle/input/**/generation_jobs.csv", recursive=True)}
print("packs found:", sorted(PACKS))
missing = [pack for _, pack, *_ in PLAN if pack not in PACKS]
if missing:
    raise SystemExit(f"Attach saikrishnareddy9/codemix-extension-bundle (Add Input); missing {missing}")

import torch
GPUS = torch.cuda.device_count()
print("GPUs:", GPUS)
if GPUS == 0:
    raise SystemExit("No GPU: set Accelerator to GPU T4 x2")

HF_TOKEN = ""
if any(tool == "indicf5" for tool, *_ in PLAN):
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN secret found")
    except Exception:
        print("No HF_TOKEN secret: IndicF5 cannot be downloaded; it will be recorded as failed.")

sh(f"{sys.executable} -m pip install -q uv")
VENVS = {}
for tool in sorted({tool for tool, *_ in PLAN}):
    venv = TEMP / "venvs" / tool
    try:
        if not (venv / "bin" / "python").exists():
            sh(f"{sys.executable} -m uv venv --python 3.11 {venv}")
        py = venv / "bin" / "python"
        sh(f"{sys.executable} -m uv pip install --python {py} -q {TORCH}")
        sh(f"{sys.executable} -m uv pip install --python {py} -q {PACKAGES[tool]}")
        VENVS[tool] = py
        print(f"environment ready: {tool}")
    except BaseException as e:
        failed(f"installing {tool}", e)

## 3. Generate, screen, bundle, and the download link

In [ ]:
import pandas as pd
from src.data import channel_bundle, portable_bundle
from src.data.generation_qa import screen, summarise

RESULTS = []
DONE = False


def launch(tool, pack, pool):
    """Run every shard of one pack on its own GPU and wait; return the exit codes."""
    if time.time() > DEADLINE:
        raise RuntimeError("the time limit was reached before this tool started")
    # MPLBACKEND: Jupyter's inline plotting backend does not exist inside the tool environments.
    env ={**os.environ, "PYTHONPATH": str(REPO), "COQUI_TOS_AGREED": "1", "HF_TOKEN": HF_TOKEN,
           "PYTHONWARNINGS": "ignore", "MPLBACKEND": "Agg"}
    procs = []
    shards = max(1, min(SHARDS, GPUS))
    for i in range(shards):
        log = open(BUNDLE / "logs" / f"{pack}_shard{i}.log", "w")
        cmd = [str(VENVS[tool]), "-m", "src.data.generate_cli", "--tool", tool,
               "--pack", str(PACKS[pack]), "--out", str(OUT / pack),
               "--pool", pool or "eval", "--shard", str(i), "--num-shards", str(shards)]
        procs.append((subprocess.Popen(cmd, cwd=REPO, env={**env, "CUDA_VISIBLE_DEVICES": str(i)},
                                       stdout=log, stderr=subprocess.STDOUT), log))
    total = len(pd.read_csv(PACKS[pack] / "generation_jobs.csv"))
    while any(p.poll() is None for p, _ in procs):
        time.sleep(120)
        made = len(list((OUT / pack).glob("*.wav"))) if (OUT / pack).exists() else 0
        print(f"  [{time.strftime('%H:%M')}] {pack}: {made}/{total} clips", flush=True)
        if time.time() > DEADLINE:
            # Stop here and bundle what exists, so the run ends (and saves) inside 12 h.
            for p, _ in procs:
                p.terminate()
            for p, _ in procs:
                try:
                    p.wait(timeout=60)
                except subprocess.TimeoutExpired:
                    p.kill()
            failed(f"{pack}: stopped at the time limit", f"{made}/{total} clips were made")
            for _, log in procs:
                log.close()
            return [0] * len(procs)  # already recorded above; the clips made are bundled
    for _, log in procs:
        log.close()
    return [p.returncode for p, _ in procs]


def bundle(pack, attack, split, language, tool):
    """Screen the clones and write the clean and phone-line bundles plus their manifests."""
    jobs = pd.read_csv(PACKS[pack] / "generation_jobs.csv")
    report = screen(jobs, str(OUT / pack))
    report.to_csv(BUNDLE / "qa" / f"{attack}_qa.csv", index=False)
    qa = summarise(report)
    ok = report[report["ok"]]
    if ok.empty:
        raise RuntimeError(f"no {pack} clip passed QA: {qa}")
    manifest = pd.DataFrame({
        "filepath": [str(OUT / pack / c) for c in ok["clip"]],
        "label": "spoof", "language": language, "speaker": ok["speaker"].astype(str).values,
        "source": attack, "tool": tool, "condition": "clean", "split": split,
        "utt_id": [f"{attack}_{Path(c).stem}" for c in ok["clip"]],
    })
    clean = portable_bundle.build(manifest, BUNDLE / "clean")
    clean.to_csv(BUNDLE / "manifests" / f"{attack}.csv", index=False)
    channel = channel_bundle.render(clean, BUNDLE / "channel", data_root=str(BUNDLE / "clean"))
    channel.to_csv(BUNDLE / "manifests" / f"{attack}_channel20.csv", index=False)
    return qa, len(clean), len(channel)


def write_summary():
    lines = ["STATUS: " + ("FINISHED - every tool ran." if DONE else
                           f"IN PROGRESS - {len(RESULTS)} set(s) finished so far."),
             "RUN HEALTH: " + ("every step ran without error." if not FAILURES else
                               f"{len(FAILURES)} step(s) FAILED -- see FAILURES.txt"),
             f"commit: {COMMIT}", ""]
    for r in RESULTS:
        lines.append(f"{r['attack']} {r['pack']:20s} generated {r['generated']:5d}  passed QA "
                     f"{r['passed']:5d}  clean {r['clean']:5d}  channel {r['channel']:5d}  "
                     f"({r['minutes']} min)")
    text = "\n".join(lines)
    (BUNDLE / "summary.txt").write_text(text, encoding="utf-8")
    (BUNDLE / "summary.json").write_text(json.dumps({
        "mode": "full" if DONE else "in progress", "commit": COMMIT, "results": RESULTS,
        "failures": FAILURES}, indent=2, default=str))
    if FAILURES:
        (BUNDLE / "FAILURES.txt").write_text(
            "\n\n".join(f"== {f['where']}\n{f['detail']}" for f in FAILURES), encoding="utf-8")
    return text


def show_download_link():
    from IPython.display import FileLink, display
    path = WORK / ZIP_NAME
    if not path.is_file():
        print("No results zip was written -- see the errors above.")
        return
    with zipfile.ZipFile(path) as zf:
        count, broken = len(zf.namelist()), zf.testzip()
    print(f"\n{ZIP_NAME}: {count} files, {path.stat().st_size / 1e6:.0f} MB,",
          "checked OK" if broken is None else f"DAMAGED at {broken}")
    os.chdir(WORK)
    print("\nCLICK THIS LINK TO DOWNLOAD THE RESULTS:")
    display(FileLink(ZIP_NAME))
    print("\nIf clicking does not download it: right-hand panel -> Output -> /kaggle/working ->",
          f"{ZIP_NAME} -> the three dots -> Download.")
    print(f"In a saved version (Save & Run All): the notebook's page -> Output -> {ZIP_NAME} -> Download.")


try:
    for tool, pack, pool, attack, split, language in PLAN:
        t0 = time.time()
        print(f"\n==================== {attack}: {tool} on {pack} ====================\n", flush=True)
        row = {"attack": attack, "pack": pack, "tool": tool, "generated": 0, "passed": 0,
               "clean": 0, "channel": 0, "minutes": 0}
        try:
            if tool not in VENVS:
                raise RuntimeError(f"the {tool} environment did not install")
            if tool == "indicf5" and not HF_TOKEN:
                raise RuntimeError("no HF_TOKEN secret; IndicF5 is gated")
            codes = launch(tool, pack, pool)
            row["generated"] = len(list((OUT / pack).glob("*.wav")))
            if any(codes):
                logs = "\n".join((BUNDLE / "logs" / f"{pack}_shard{i}.log").read_text()[-1500:]
                                 for i in range(len(codes)))
                failed(f"{pack}: a generation process exited {codes}", logs)
            if row["generated"]:
                qa, row["clean"], row["channel"] = bundle(pack, attack, split, language, tool)
                row["passed"] = int(qa.get("passed", row["clean"]))
        except Exception:
            failed(f"{attack} ({pack})", traceback.format_exc())
        finally:
            row["minutes"] = round((time.time() - t0) / 60, 1)
            RESULTS.append(row)
            write_summary()
            save_progress()
            print(f"\n[{attack}] generated {row['generated']}, in bundle {row['clean']}  "
                  f"({row['minutes']} min)\n", flush=True)
    DONE = True
    print("\nALL TOOLS FINISHED\n", flush=True)
finally:
    print(write_summary(), flush=True)
    save_progress()
    show_download_link()

## 4. Optional: the download link again

Section 3 already ends with the link. Run this only if you need it again.

In [ ]:
import os, zipfile
from pathlib import Path
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
if Path(ZIP_NAME).is_file():
    with zipfile.ZipFile(ZIP_NAME) as zf:
        print(f"{ZIP_NAME}: {len(zf.namelist())} files, checked", "OK" if zf.testzip() is None else "DAMAGED")
    display(FileLink(ZIP_NAME))
else:
    print("No results zip yet -- section 3 writes it after its first finished tool.")